# Final Data Science Capstone Project
## Breast Cancer Diagnostic Triage Support
**Data Science Consultant Internship — Task 6**

End-to-end capstone: problem definition, data preparation, EDA, machine learning (two models compared), evaluation and interpretation, visualizations, recommendations, and future scope.

*Requires `breast_cancer_clean.csv` (the cleaned dataset from Task 3) in the same folder.*

## 1. Define the Capstone Problem
**Problem:** clinical teams need a fast, interpretable way to flag likely-malignant breast tumors from routine cell measurements so that cases can be prioritized for closer review.

**Objectives:** (1) find which measurements best separate malignant from benign cases, (2) build and compare classifiers, (3) recommend a practical triage approach.

**Scope:** 569 cases, 8 mean-value tumor measurements. This is decision *support* on a historical benchmark dataset, not a diagnostic tool.

**Success criteria:** high recall on malignant cases (missing a malignant case is the costly error), with overall accuracy and ROC-AUC as supporting metrics.

## 2. Collect & Prepare Data

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

df = pd.read_csv('breast_cancer_clean.csv')   # cleaned in Task 3: duplicates removed, missing values imputed, labels standardized
print(df.shape)
print("Missing values:", df.isnull().sum().sum(), "| Duplicates:", df.duplicated().sum())
df.head()

*Data note:* the source (Breast Cancer Wisconsin Diagnostic, via scikit-learn) is already clean. In Task 3, a few missing values, inconsistent labels and duplicate rows were deliberately introduced and then cleaned, so the cleaning workflow could be demonstrated.

In [ ]:
# Feature engineering (from Task 3) is already in the file: area_to_perimeter, compactness_ratio, radius_category
feats = ['radius_mean','texture_mean','perimeter_mean','area_mean','smoothness_mean',
         'compactness_mean','concavity_mean','symmetry_mean']
print(df['diagnosis'].value_counts())
df.groupby('diagnosis')[feats].mean().round(2)

### Exploratory data analysis

In [ ]:
plt.figure(figsize=(7,5))
for d, col in {'benign':'#55A868','malignant':'#C44E52'}.items():
    sns.histplot(df[df['diagnosis']==d]['radius_mean'], bins=25, color=col, label=d, alpha=0.6)
plt.title('Distribution of Mean Radius by Diagnosis'); plt.legend(); plt.show()

In [ ]:
plt.figure(figsize=(7,6))
sns.heatmap(df[feats].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, annot_kws={'size':8})
plt.title('Correlation Heatmap of Tumor Features'); plt.show()

In [ ]:
order = ['Small','Medium','Large','Very Large']
rate = (df.assign(m=(df['diagnosis']=='malignant')*100)
          .groupby('radius_category')['m'].mean().reindex(order).round(1))
plt.figure(figsize=(7,5)); plt.bar(rate.index, rate.values, color='#DD8452')
plt.ylabel('Malignant rate (%)'); plt.title('Malignancy Rate by Tumor Size Category'); plt.show()
rate

## 3. Apply Data Science Techniques
Two classifiers are compared: **Logistic Regression** (interpretable baseline) and **Random Forest** (non-linear). Data is split 80/20 (stratified); 5-fold stratified cross-validation is run on the training set.

In [ ]:
X = df[feats]
y = (df['diagnosis'] == 'malignant').astype(int)   # 1 = malignant
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)

rows, probs, cms = [], {}, {}
for name, m in models.items():
    cv_recall = cross_val_score(m, Xtr, ytr, cv=cv, scoring='recall')
    m.fit(Xtr, ytr)
    pred = m.predict(Xte); prob = m.predict_proba(Xte)[:, 1]
    probs[name] = prob; cms[name] = confusion_matrix(yte, pred)
    rows.append({'Model': name,
                 'Accuracy': accuracy_score(yte, pred), 'Precision': precision_score(yte, pred),
                 'Recall': recall_score(yte, pred), 'F1': f1_score(yte, pred),
                 'ROC-AUC': roc_auc_score(yte, prob),
                 'CV Recall (mean)': cv_recall.mean(), 'CV Recall (std)': cv_recall.std()})
results = pd.DataFrame(rows).set_index('Model').round(4)
results

## 4. Evaluate & Interpret Results

In [ ]:
# ROC curves
plt.figure(figsize=(6.5,5.5))
for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(yte, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(yte, prob):.3f})")
plt.plot([0,1],[0,1],'k--', alpha=.4)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curves'); plt.legend(); plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
for ax, (name, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['benign','malignant'],
                yticklabels=['benign','malignant'], ax=ax)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

In [ ]:
# Interpretation: standardized logistic coefficients and random forest importances
lr = models['Logistic Regression']
coefs = pd.Series(lr.named_steps['logisticregression'].coef_[0], index=feats).sort_values()
imp = pd.Series(models['Random Forest'].feature_importances_, index=feats).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(13,5))
axes[0].barh(coefs.index, coefs.values, color=['#C44E52' if v<0 else '#4C72B0' for v in coefs.values])
axes[0].set_title('Logistic Regression coefficients (standardized)')
axes[1].barh(imp.index, imp.values, color='#4C72B0'); axes[1].set_title('Random Forest feature importance')
plt.tight_layout(); plt.show()

**Reading the results**
- Both models perform similarly: ~93-94% test accuracy and ROC-AUC ≈ 0.98. Random Forest has slightly higher accuracy/precision; Logistic Regression has slightly higher recall on malignant cases (92.9% vs 90.5% on the test set).
- With only 42 malignant cases in the test set, one case changes recall by ~2.4 points, so these differences are **not strong evidence** that one model is better.
- Cross-validated recall (~88%) is lower than test-set recall, which suggests the test-set figure may be slightly optimistic. Expect roughly 88-93% recall on new data from the same source.
- Both models agree that radius, perimeter and area (size) are the dominant predictors, with concavity next.

## 5. Visualizations & Recommendations

1. **Use a two-stage triage:** flag by tumor size first; send medium-size, ambiguous cases to the model and a human reviewer.
2. **Prefer Logistic Regression for deployment** unless more data shows Random Forest is clearly better: comparable performance, higher test recall, and coefficients that clinical staff can interpret.
3. **Track recall on malignant cases**, not accuracy alone, and tune the decision threshold toward catching more malignant cases if more false alarms are acceptable.
4. **Keep a human in the loop.** The model supports prioritization; it does not replace diagnosis.

## 6. Limitations & Future Scope
**Limitations:** small, historical benchmark dataset (569 cases); small test set (114 cases, 42 malignant); no external validation; only mean-value features were used; no hyperparameter tuning.

**Future scope:** use all 30 original features; tune hyperparameters and decision thresholds; add calibration and more models (e.g. gradient boosting); validate on newer data from other sources; build the Power BI dashboard specified in Task 4 for stakeholder monitoring.

See the full project report in `Final_Capstone_Project_Report.docx`.